In [2]:
from azureml.core import Workspace, Dataset
from azureml.datadrift import DataDriftDetector

from datetime import datetime, timedelta

In [4]:
ws     = Workspace.from_config()
dstore = ws.get_default_datastore()

In [5]:
dstore.upload('weather-data', 'florida-weather', overwrite=True)

Uploading an estimated of 118 files
Uploading weather-data/2010/01/data.parquet
Uploading weather-data/2010/02/data.parquet
Uploading weather-data/2010/03/data.parquet
Uploading weather-data/2010/04/data.parquet
Uploading weather-data/2010/05/data.parquet
Uploading weather-data/2010/06/data.parquet
Uploading weather-data/2010/07/data.parquet
Uploading weather-data/2010/08/data.parquet
Uploading weather-data/2010/09/data.parquet
Uploading weather-data/2010/10/data.parquet
Uploading weather-data/2010/11/data.parquet
Uploading weather-data/2010/12/data.parquet
Uploading weather-data/2011/01/data.parquet
Uploading weather-data/2011/02/data.parquet
Uploading weather-data/2011/03/data.parquet
Uploading weather-data/2011/04/data.parquet
Uploading weather-data/2011/05/data.parquet
Uploading weather-data/2011/06/data.parquet
Uploading weather-data/2011/07/data.parquet
Uploading weather-data/2011/08/data.parquet
Uploading weather-data/2011/09/data.parquet
Uploading weather-data/2011/10/data.parq

$AZUREML_DATAREFERENCE_cf0e7d024c274e0bb0b57ff15a2d146f

In [6]:
dset   = Dataset.Tabular.from_parquet_files(dstore.path('florida-weather/**/data.parquet'))

In [7]:
target = dset.with_timestamp_columns('datetime')
target = target.register(ws, 'target')

In [8]:
target = Dataset.get_by_name(ws, 'target')

In [9]:
baseline = target.time_recent(timedelta(weeks=4))

In [10]:
features = ['latitude', 'longitude', 'elevation', 'windAngle', 'windSpeed', 'temperature', 'snowDepth', 'stationName', 'countryOrRegion']

In [15]:
monitor = DataDriftDetector.create_from_datasets(ws, 'weatherMonitor3', baseline, target, 
                                                      compute_target_name='cpu-cluster', 
                                                      frequency='Week', 
                                                      feature_list=None, 
                                                      drift_threshold=None, 
                                                      latency=0)

In [16]:
monitor = DataDriftDetector.get_by_name(ws, 'weatherMonitor3')

monitor = monitor.update(drift_threshold=.4)
monitor = monitor.update(feature_list=features)

In [17]:
for year in range(2010, 2020): 
    monitor.backfill(datetime(year, 1, 1), datetime(year, 7, 1))
    monitor.backfill(datetime(year, 7, 1), datetime(year+1, 1, 1))

In [14]:
help(monitor)

Help on DataDriftDetector in module azureml.datadrift.datadriftdetector object:

class DataDriftDetector(builtins.object)
 |  DataDriftDetector(workspace, model_name, model_version)
 |  
 |  Class for AzureML DataDriftDetector.
 |  
 |  DataDriftDetector class provides set of convenient APIs to identify any drifting between given training
 |  and/or scoring datasets for a model. A DataDriftDetector object is created with a workspace, a model name
 |  and version, list of services, and optional tuning parameters. A DataDriftDetector task can be scheduled
 |  using enable_schedule() API and/or a one time task can be submitted using run() method.
 |  
 |  Methods defined here:
 |  
 |  __init__(self, workspace, model_name, model_version)
 |      Datadriftdetector constructor.
 |      
 |      The DataDriftDetector constructor is used to retrieve a cloud representation of a DataDriftDetector object
 |      associated with the provided workspace. Must provide model_name and model_version.
 